# KLEOS 03 — Evaluate and compareRun the base and fine-tuned arms under **identical conditions** and compare themhonestly.The comparison reports per-task deltas with bootstrap confidence intervals, plusOOD and consistency separately. An improvement that is not statisticallysignificant is reported as *not significant*, not as a win.

## 1. Setup

In [ ]:
# Clone the repository (skip if already present) and enter it.import osfrom pathlib import PathREPO_DIR = Path("/content/kleos-models")if not REPO_DIR.exists():    !git clone https://github.com/kleos/kleos-models.git {REPO_DIR}else:    print(f"{REPO_DIR} already exists; pulling latest")    !cd {REPO_DIR} && git pull --ff-onlyos.chdir(REPO_DIR)print("working directory:", Path.cwd())

In [ ]:
# Install dependencies WITHOUT touching Colab's torch build.## Reinstalling torch on Colab replaces the build compiled against this runtime's# CUDA driver, and CUDA then silently stops working. scripts/colab_setup.py uses# --no-deps for every package that would otherwise pull torch along.!python scripts/colab_setup.py

In [ ]:
# Hugging Face authentication.## Needed only for gated base models (Mistral) or to publish an adapter.# Use Colab Secrets (the key icon in the left sidebar), never a literal token in# a cell — notebooks get shared and committed.import ostry:    from google.colab import userdata    token = userdata.get("HF_TOKEN")    if token:        os.environ["HF_TOKEN"] = token        print("HF_TOKEN loaded from Colab secrets.")    else:        print("No HF_TOKEN secret set.")except Exception as exc:    print(f"Colab secrets unavailable ({type(exc).__name__}).")    print("Set os.environ['HF_TOKEN'] manually if you need gated models.")if not os.environ.get("HF_TOKEN"):    print()    print("Without a token you can still use ungated models such as Qwen/Qwen3-8B.")    print("To add one: sidebar key icon -> Add new secret -> name HF_TOKEN ->")    print("enable 'Notebook access'.")

In [ ]:
# What GPU did Colab actually assign? Free-tier allocation varies.!nvidia-smifrom kleos_models.models.feasibility import probe_gpugpu = probe_gpu()print()print(gpu.render())print()if not gpu.available:    print("NO GPU ASSIGNED.")    print("Runtime -> Change runtime type -> T4 GPU, then re-run this cell.")elif not gpu.bf16_supported:    print(f"Note: {gpu.name} (compute capability {gpu.capability_string}) has no")    print("bfloat16 support. KLEOS configs use compute_dtype: auto, which selects")    print("float16 here automatically. Nothing to change.")

## 2. Point at the run to evaluate

In [ ]:
CONFIG = "configs/training/qlora_small.yaml"BENCHMARK = "data/examples/synthetic_eval.jsonl"USE_DRIVE = Trueif USE_DRIVE:    from google.colab import drive    drive.mount("/content/drive")    OUTPUT_DIR = "/content/drive/MyDrive/kleos/outputs"else:    OUTPUT_DIR = "outputs"from kleos_models.experiments.registry import ExperimentRegistryregistry = ExperimentRegistry(OUTPUT_DIR)runs = registry.filter(kind="training")if not runs:    raise SystemExit(f"No training runs found under {OUTPUT_DIR}. Run notebook 02 first.")latest = runs[0]ADAPTER = str(latest.directory / "adapter")print(latest.manifest.summary())print()print("adapter:", ADAPTER)

## 3. Evaluate the base model (arm 0)No adapter. This is the baseline the fine-tuned arm must beat.

In [ ]:
!python scripts/evaluate.py \    --config {CONFIG} \    --arm arm0_base \    --benchmark {BENCHMARK} \    --output {OUTPUT_DIR}/base_results.json

## 4. Evaluate the fine-tuned model (arm 2)Identical base weights, identical decoding, identical benchmark. **The adapter isthe only difference** — that is what makes the comparison meaningful.

In [ ]:
!python scripts/evaluate.py \    --config {CONFIG} \    --arm arm2_finetuned \    --adapter {ADAPTER} \    --benchmark {BENCHMARK} \    --output {OUTPUT_DIR}/finetuned_results.json

## 5. Compare

In [ ]:
!python scripts/compare.py \    --base {OUTPUT_DIR}/base_results.json \    --finetuned {OUTPUT_DIR}/finetuned_results.json \    --report reports/latest

### How to read the comparison- **Per task, not blended.** A single aggregate would hide a model that improves  one task while harming another — which may be the most interesting result  available.- **Significance.** A positive delta marked *(not significant)* is not a win. It  means the evaluation set is too small or too noisy to tell.- **OOD delta.** An in-distribution gain with an OOD loss is consistent with  fitting surface features rather than learning a transferable policy.- **Consistency delta.** Whether the model reaches the same decision under  logically irrelevant perturbations. This is the clearest signal of a learned  policy versus a surface pattern.- **Where it failed.** Always read this section.If OOD is reported as *not measured*, no generalization claim can be made fromthis run — tag benchmark examples with `split_tag: "ood"` to enable it.

## 6. Read the report

In [ ]:
from pathlib import Pathfrom IPython.display import Markdown, displaysummary = Path("reports/latest/summary.md")if summary.exists():    display(Markdown(summary.read_text()))else:    print("No report found; check that section 5 completed.")

## A note on interpreting resultsIf fine-tuning wins, quantify the win and check it survives OOD.If it loses, that is a valid, publishable result — analyse data quality, policylearnability, capacity and evaluation design.If it improves one task and harms another, that is likely the most informativeoutcome the experiment can produce.Record the run either way. `docs/experiments.md` holds the pre-registeredhypotheses this evaluation is meant to test.